<div align="center">

# Redes Informadas por la Física (PINNs)

### Solución de ecuaciones diferenciales ordinarias: problemas directos e inversos

**Lenin Herrera Padilla** · lenin.herrera.p@uni.pe

</div>

---

Este notebook reúne el código de los casos de estudio: la comparación entre una red neuronal estándar y una PINN, la elección de la función de activación, problemas directos e inversos, y los casos donde la PINN aporta frente a los métodos clásicos, junto con sus límites.

| Bloque | Contenido |
|---|---|
| 0 | Configuración del entorno |
| 1 | Motivación: el problema de las EDO |
| 2 | Red neuronal estándar |
| 3 | PINN: la física dentro de la función de pérdida |
| 4 | Elección de la función de activación |
| 5 | Problemas directos: ecuación logística y oscilador armónico simple |
| 6 | Problemas inversos: gripe, oscilador con datos de un simulador y sistema de Lorenz |
| 7 | Comparación, límites y conclusiones |

La extensión a ecuaciones en derivadas parciales está en el notebook `PINNs_EDP.ipynb`.

**Entorno:** Google Colab · Python 3 · PyTorch, DeepXDE, SciPy, NumPy, Matplotlib.

> Ejecutar las celdas en orden (*Entorno de ejecución → Ejecutar todas*). Con `RAPIDO = True` (celda siguiente) todos los entrenamientos se reducen unas diez veces, útil para una demostración rápida; los resultados completos se obtienen con `RAPIDO = False`.

## 0. Configuración del entorno

In [ ]:
# --- modo de ejecución -------------------------------------------------
RAPIDO = False        # True: entrenamientos cortos para una demostración en vivo

def IT(n):
    # número de iteraciones según el modo de ejecución
    return max(n // 10, 300) if RAPIDO else n

# --- instalación de DeepXDE (solo si no está disponible) ----------------
import importlib.util, subprocess, sys
if importlib.util.find_spec("deepxde") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepxde"], check=True)

import os
os.environ["DDE_BACKEND"] = "pytorch"     # DeepXDE con PyTorch

import math, time, re
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import deepxde as dde

if RAPIDO:
    dde.optimizers.config.set_LBFGS_options(maxiter=1000)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
AZUL, ROJO, VERDE, GRIS, NARANJA = "#1F5AA6", "#C0392B", "#2E8B57", "#7F7F7F", "#E08E0B"

def error_l2(aprox, exacta):
    # error L2 relativo
    aprox, exacta = np.ravel(aprox), np.ravel(exacta)
    return np.linalg.norm(aprox - exacta) / np.linalg.norm(exacta)

def leer(v):
    # valor numérico de una variable entrenable de DeepXDE
    try:
        return float(v.detach())
    except AttributeError:
        return float(v)

print("PyTorch", torch.__version__, "| DeepXDE", dde.__version__, "| modo rápido:", RAPIDO)

## 1. Motivación: el problema de las EDO

Todo fenómeno que cambia en el tiempo según una ley se modela con una ecuación diferencial ordinaria:

| Campo | Ecuación | Aplicación |
|---|---|---|
| Ingeniería | $L\,q''+R\,q'+\tfrac{1}{C}\,q=E(t)$ | circuitos RLC, vibraciones, control |
| Economía | $k'=s\,f(k)-(n+\delta)\,k$ | crecimiento (Solow), adopción de tecnologías |
| Salud | $I'=\beta SI-\gamma I$ | epidemias (SIR), eliminación de fármacos |
| Ecología | $x'=\alpha x-\beta xy$ | presas y depredadores (Lotka-Volterra) |
| Física | $m\,y''+b\,y'+k\,y=0$ | osciladores, caída libre, calor |
| Clima | $\mathbf{u}'=\mathbf{F}(\mathbf{u};\sigma,\rho,\beta)$ | convección atmosférica (Lorenz) |

**Problema de valor inicial:** $\dfrac{du}{dt}=f\big(t,u(t)\big)$, $t\in[t_0,T]$, $u(t_0)=u_0$.

* **Problema directo:** se conocen $f$ y sus parámetros; se busca $u(t)$.
* **Problema inverso:** se conocen observaciones $u(t_i)$; se buscan los parámetros de $f$.

**Idea:** aproximar la solución por una red neuronal, $u(t)\approx u_\theta(t)$, y exigir que cumpla la ecuación.

## 2. Red neuronal estándar

**Experimento:** oscilador amortiguado con pocos datos,

$$x''+2\delta x'+\omega_0^2x=0,\qquad x(0)=1,\quad x'(0)=0,\qquad \delta=2,\ \omega_0=20,$$

con solución exacta $x(t)=1{,}00504\,e^{-2t}\cos(19{,}89975\,t-0{,}10017)$. Solo se observan **10 puntos**, todos en $t\in[0;\,0{,}4]$.

*El planteamiento y el código base de esta comparación son una adaptación del trabajo de Ben Moseley (2022); la solución analítica sigue la derivación de Berg.*

In [ ]:
def oscilador(d, w0, t):
    # solución analítica del oscilador amortiguado (régimen subamortiguado)
    w = np.sqrt(w0**2 - d**2)
    phi = np.arctan(-d / w)
    A = 1 / (2 * np.cos(phi))
    return torch.exp(-d * t) * 2 * A * torch.cos(phi + w * t)

class RedTotalmenteConectada(nn.Module):
    # red prealimentada totalmente conectada con activación tanh
    def __init__(self, n_entrada, n_salida, n_ocultas, n_capas):
        super().__init__()
        self.capa_entrada = nn.Sequential(nn.Linear(n_entrada, n_ocultas), nn.Tanh())
        self.capas_ocultas = nn.Sequential(*[
            nn.Sequential(nn.Linear(n_ocultas, n_ocultas), nn.Tanh()) for _ in range(n_capas - 1)])
        self.capa_salida = nn.Linear(n_ocultas, n_salida)
    def forward(self, t):
        return self.capa_salida(self.capas_ocultas(self.capa_entrada(t)))

d, w0 = 2, 20
x = torch.linspace(0, 1, 500).view(-1, 1)          # dominio completo
y = oscilador(d, w0, x).view(-1, 1)                # solución exacta
x_datos, y_datos = x[0:200:20], y[0:200:20]        # 10 observaciones en [0; 0,4]
CORTE = 0.4                                        # frontera de la región con datos

plt.figure(figsize=(7, 3.6))
plt.plot(x, y, color=GRIS, label="Solución exacta")
plt.scatter(x_datos, y_datos, color=NARANJA, zorder=5, label="Datos de entrenamiento")
plt.axvspan(CORTE, 1, color="0.93")
plt.xlabel("t"); plt.ylabel("x(t)"); plt.legend(); plt.title("Oscilador amortiguado: 10 observaciones")
plt.show()

**Entrenamiento solo con datos:** $\;\mathcal{L}_{\text{datos}}(\theta)=\dfrac{1}{N_d}\displaystyle\sum_{i=1}^{N_d}\big|u_\theta(t_i)-u_i\big|^2$, con Adam ($\eta=10^{-4}$).

**Arquitectura:** $1-32-32-32-1$ · $\tanh$ · Adam, $20\,000$ iteraciones.

In [ ]:
EPOCAS = [1000, 5000, 10000, 20000]
N_IT = IT(20000)
EPOCAS_GUARDADAS = sorted(set(min(e, N_IT) for e in EPOCAS))

# --- red estándar 1-32-32-32-1: solo datos ---
torch.manual_seed(123)
red_estandar = RedTotalmenteConectada(1, 1, 32, 3)
opt = torch.optim.Adam(red_estandar.parameters(),
                       lr=1e-4)
evol_estandar = {}
for i in range(N_IT):
    opt.zero_grad()
    perdida = torch.mean(
        (red_estandar(x_datos) - y_datos) ** 2)
    perdida.backward()
    opt.step()
    if i + 1 in EPOCAS_GUARDADAS:
        evol_estandar[i + 1] = red_estandar(x).detach()

yh_estandar = red_estandar(x).detach()
mc, ms = x.view(-1) <= CORTE, x.view(-1) > CORTE
print(f"Red estándar: L2 con datos = {error_l2(yh_estandar[mc], y[mc]):.3e} | "
      f"L2 sin datos = {error_l2(yh_estandar[ms], y[ms]):.3e}")

## 3. PINN: la física dentro de la función de pérdida

**Residuo:** $r_\theta(t)=u_\theta''+2\delta\,u_\theta'+\omega_0^2\,u_\theta$, evaluado por **diferenciación automática** en 30 puntos de colocación.

$$\mathcal{L}(\theta)=\underbrace{\frac{1}{N_d}\sum_{i=1}^{N_d}\big|u_\theta(t_i)-u_i\big|^2}_{\mathcal{L}_{\text{datos}}}+\;\omega_F\underbrace{\frac{1}{N_c}\sum_{j=1}^{N_c}\big|r_\theta(t_j)\big|^2}_{\mathcal{L}_F},\qquad \omega_F=10^{-4}.$$

**Arquitectura:** idéntica a la red estándar ($1-32-32-32-1$ · $\tanh$ · Adam, $20\,000$ iteraciones). **La única diferencia está en la pérdida.**

In [ ]:
# --- PINN: misma red, semilla y optimizador; cambia la pérdida ---
x_fisica = torch.linspace(0, 1, 30).view(-1, 1)
x_fisica.requires_grad_(True)
mu, k = 2 * d, w0 ** 2
torch.manual_seed(123)
pinn = RedTotalmenteConectada(1, 1, 32, 3)
opt = torch.optim.Adam(pinn.parameters(), lr=1e-4)
evol_pinn = {}
for i in range(N_IT):
    opt.zero_grad()
    p_datos = torch.mean((pinn(x_datos) - y_datos) ** 2)
    yhp = pinn(x_fisica)
    dx = torch.autograd.grad(yhp, x_fisica,
            torch.ones_like(yhp), create_graph=True)[0]
    dx2 = torch.autograd.grad(dx, x_fisica,
            torch.ones_like(dx), create_graph=True)[0]
    residuo = dx2 + mu * dx + k * yhp
    p_fisica = 1e-4 * torch.mean(residuo ** 2)
    (p_datos + p_fisica).backward()
    opt.step()
    if i + 1 in EPOCAS_GUARDADAS:
        evol_pinn[i + 1] = pinn(x).detach()

yh_pinn = pinn(x).detach()
print(f"PINN:         L2 con datos = {error_l2(yh_pinn[mc], y[mc]):.3e} | "
      f"L2 sin datos = {error_l2(yh_pinn[ms], y[ms]):.3e}")

### Evolución de ambas redes: la red estándar interpola, la PINN extrapola

In [ ]:
fig, ax = plt.subplots(2, len(EPOCAS_GUARDADAS), figsize=(12, 5.2), sharex=True, sharey=True, squeeze=False)
for r, (evol, color, nombre) in enumerate([(evol_estandar, ROJO, "Red estándar"), (evol_pinn, AZUL, "PINN")]):
    for c, ep in enumerate(EPOCAS_GUARDADAS):
        a = ax[r, c]
        a.axvspan(CORTE, 1, color="0.93")
        a.plot(x, y, color="0.6", lw=2.5)
        a.plot(x, evol[ep], "--", color=color, lw=1.8)
        a.plot(x_datos, y_datos, "o", color=NARANJA, ms=4)
        a.set_ylim(-1.3, 1.3); a.set_xlim(0, 1)
        if r == 0: a.set_title(f"{ep} épocas")
        if c == 0: a.set_ylabel(f"{nombre}\nx(t)")
        if r == 1: a.set_xlabel("t")
fig.suptitle("Zona sombreada: sin datos"); fig.tight_layout(); plt.show()

### Comparación final y error puntual

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
a = ax[0]
a.plot(x, y, color=GRIS, lw=2.5, alpha=0.7, label="Solución exacta")
a.plot(x, yh_estandar, "--", color=ROJO, lw=1.8, label="Red estándar")
a.plot(x, yh_pinn, "-.", color=AZUL, lw=1.8, label="PINN")
a.scatter(x_datos, y_datos, s=40, color=NARANJA, zorder=5, label="Datos")
a.axvspan(CORTE, 1, color="0.93"); a.set_xlabel("t"); a.set_ylabel("x(t)")
a.legend(fontsize=9); a.set_title("(a) Soluciones obtenidas")
a = ax[1]
a.semilogy(x, np.abs(yh_estandar - y) + 1e-12, color=ROJO, label="Red estándar")
a.semilogy(x, np.abs(yh_pinn - y) + 1e-12, color=AZUL, label="PINN")
a.axvspan(CORTE, 1, color="0.93"); a.set_xlabel("t"); a.set_ylabel("Error absoluto")
a.legend(fontsize=9); a.set_title("(b) Error puntual (escala log)")
fig.tight_layout(); plt.show()

### ¿Qué es la función aproximada?

La solución de la PINN no es una fórmula cerrada: es la red misma, una composición de multiplicaciones matriciales y activaciones $\tanh$ gobernada por sus parámetros.

In [ ]:
total = 0
for nombre, p in pinn.named_parameters():
    total += p.numel()
    print(f"  {nombre:<28} forma {str(tuple(p.shape)):<10} -> {p.numel()} valores")
print(f"\nTotal de parámetros ajustables: {total}")

# la PINN entrenada es una función continua: puede evaluarse en cualquier t
t_eval = 0.73
print(f"\nu_theta({t_eval}) = {pinn(torch.tensor([[t_eval]])).item():.6f}   "
      f"exacta = {oscilador(d, w0, torch.tensor([[t_eval]])).item():.6f}")

### Evolución de la PINN por iteraciones

Se reentrena la PINN y se guardan instantáneas a las 500, 3 000, 10 000 y 30 000 iteraciones.

In [ ]:
claves = sorted(set(IT(n) for n in [500, 3000, 10000, 30000]))
instantaneas = {}
torch.manual_seed(123)
pinn_prog = RedTotalmenteConectada(1, 1, 32, 3)
opt = torch.optim.Adam(pinn_prog.parameters(), lr=1e-4)
for i in range(1, max(claves) + 1):
    opt.zero_grad()
    p_datos = torch.mean((pinn_prog(x_datos) - y_datos) ** 2)
    yhp = pinn_prog(x_fisica)
    dx  = torch.autograd.grad(yhp, x_fisica, torch.ones_like(yhp), create_graph=True)[0]
    dx2 = torch.autograd.grad(dx,  x_fisica, torch.ones_like(dx),  create_graph=True)[0]
    (p_datos + 1e-4 * torch.mean((dx2 + mu * dx + k * yhp) ** 2)).backward()
    opt.step()
    if i in claves:
        instantaneas[i] = pinn_prog(x).detach()
        print(f"iteración {i:>6}:  L2 relativo = {error_l2(instantaneas[i], y):.3e}")

fig, axes = plt.subplots(2, 2, figsize=(11, 6.2))
for a, it in zip(axes.ravel(), claves):
    a.plot(x, y, color=GRIS, lw=2, alpha=0.7, label="Solución exacta")
    a.plot(x, instantaneas[it], color=AZUL, lw=2.3, label="PINN")
    a.scatter(x_datos, y_datos, s=30, color=NARANJA, zorder=5, label="Datos")
    a.scatter(x_fisica.detach(), -0.02 * torch.ones_like(x_fisica.detach()), s=18, color=VERDE, label="Colocación")
    a.set_title(f"Iteración {it}  (L2 = {error_l2(instantaneas[it], y):.2e})", fontsize=10)
    a.set_xlim(0, 1); a.set_ylim(-1.15, 1.15); a.set_xlabel("t"); a.set_ylabel("x(t)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Evolución de la PINN durante el entrenamiento"); fig.tight_layout(); plt.show()

## 4. Elección de la función de activación

La derivada de orden $k$ de la red hereda $\sigma^{(k)}$: una EDO de orden $n$ requiere $\sigma\in C^{n}$ con $\sigma^{(k)}\not\equiv0$ para $k\le n$.

* $\tanh\in C^\infty$: $\;\sigma''=-2\tanh x\,\mathrm{sech}^2x\not\equiv0$.
* $\mathrm{ReLU}$: $\;\sigma''=0$ en casi todo punto; el residuo de segundo orden pierde $u_\theta''$.

In [ ]:
activaciones = [("Sigmoide", nn.Sigmoid(), "#9467bd"), ("Tangente hiperbólica", nn.Tanh(), AZUL),
                ("ReLU", nn.ReLU(), ROJO), ("SiLU", nn.SiLU(), "#17becf"),
                ("GELU", nn.GELU(), VERDE), ("Softplus", nn.Softplus(), NARANJA)]
xa = torch.linspace(-4, 4, 800).reshape(-1, 1).requires_grad_(True)
xn = xa.detach().numpy().ravel()

fig, ax = plt.subplots(len(activaciones), 2, figsize=(9.5, 12))
for i, (nombre, f, c) in enumerate(activaciones):
    ya = f(xa)
    dy = torch.autograd.grad(ya, xa, torch.ones_like(ya))[0]
    ax[i, 0].plot(xn, ya.detach().numpy().ravel(), color=c, lw=2.3)
    ax[i, 1].plot(xn, dy.detach().numpy().ravel(), color=c, lw=2.3, ls="--")
    ax[i, 0].set_ylabel(nombre, fontweight="bold")
    if i == 0:
        ax[i, 0].set_title("Función  σ(x)"); ax[i, 1].set_title("Derivada  σ′(x)")
fig.tight_layout(); plt.show()

In [ ]:
# tanh frente a seno: función, segunda y tercera derivada
xs = np.linspace(-5, 5, 1000); th = np.tanh(xs); s2 = 1 / np.cosh(xs) ** 2
paneles = [(th, np.sin(xs), "(a) Función"),
           (-2 * th * s2, -np.sin(xs), "(b) Segunda derivada"),
           (-2 * s2 * (1 - 3 * th ** 2), -np.cos(xs), "(c) Tercera derivada")]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (yt, ys, titulo) in zip(ax, paneles):
    a.plot(xs, yt, color=AZUL, lw=2, label="tanh(x)")
    a.plot(xs, ys, "--", color=ROJO, lw=2, label="sen(x)")
    a.set_title(titulo); a.set_xlabel("x")
ax[0].legend(); fig.tight_layout(); plt.show()
# las derivadas de tanh se anulan lejos del origen; las del seno no

### Oscilador de alta frecuencia: $\tanh$ frente a seno (SIREN)

$$u''+\omega^2u=0,\qquad u(0)=5,\ u'(0)=0\quad\Rightarrow\quad u(t)=5\cos\omega t,\qquad \omega=k\pi,\ t\in[0,1].$$

**Arquitectura:** $1-64-64-64-64-1$ · $\tanh$ o seno · Adam $12\,000$ (tasa escalonada). La red SIREN usa la inicialización de Sitzmann et al. (2020).

In [ ]:
torch.set_default_dtype(torch.float64)        # doble precisión solo para este experimento
def fijar_semilla(s=0): torch.manual_seed(s); np.random.seed(s)

class RedTanh(nn.Module):
    def __init__(self, ancho=64, prof=4):
        super().__init__(); L = [nn.Linear(1, ancho), nn.Tanh()]
        for _ in range(prof - 1): L += [nn.Linear(ancho, ancho), nn.Tanh()]
        L += [nn.Linear(ancho, 1)]; self.net = nn.Sequential(*L)
    def forward(self, t): return self.net(t)

class Seno(nn.Module):
    def __init__(self, w0=1.0): super().__init__(); self.w0 = w0
    def forward(self, x): return torch.sin(self.w0 * x)

class SIREN(nn.Module):
    # red senoidal con la inicialización de Sitzmann et al. (2020)
    def __init__(self, ancho=64, prof=4, w0_primera=30.0, w0=1.0):
        super().__init__(); self.capas = nn.ModuleList(); self.acts = nn.ModuleList()
        dims = [1] + [ancho] * prof
        for i in range(prof):
            self.capas.append(nn.Linear(dims[i], dims[i + 1]))
            self.acts.append(Seno(w0_primera if i == 0 else w0))
        self.salida = nn.Linear(ancho, 1)
        with torch.no_grad():
            for i, l in enumerate(self.capas):
                fan = l.weight.shape[1]
                if i == 0: l.weight.uniform_(-1 / fan, 1 / fan)
                else: l.weight.uniform_(-math.sqrt(6 / fan) / w0, math.sqrt(6 / fan) / w0)
                l.bias.uniform_(-1 / math.sqrt(fan), 1 / math.sqrt(fan))
            fan = self.salida.weight.shape[1]
            self.salida.weight.uniform_(-math.sqrt(6 / fan), math.sqrt(6 / fan))
    def forward(self, t):
        h = t
        for l, a in zip(self.capas, self.acts): h = a(l(h))
        return self.salida(h)

AMP = 5.0
def deriv(u, t): return torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]

def entrenar_osc(modelo, w, it=12000, N=400, lr=2e-3):
    it = IT(it)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.StepLR(opt, max(it // 3, 1), 0.5)
    tc = torch.linspace(0, 1, N).reshape(-1, 1).requires_grad_(True)
    t0 = torch.zeros(1, 1).requires_grad_(True)
    for _ in range(it):
        opt.zero_grad()
        u = modelo(tc); res = (deriv(deriv(u, tc), tc) + w ** 2 * u) / w ** 2      # residuo normalizado
        ci = (modelo(t0) - AMP) ** 2 + (deriv(modelo(t0), t0) / w) ** 2            # u(0)=5, u'(0)=0
        (res.pow(2).mean() + 50.0 * ci).backward(); opt.step(); sch.step()
    tt = torch.linspace(0, 1, 600).reshape(-1, 1)
    with torch.no_grad(): up = modelo(tt).numpy().ravel()
    tn = tt.numpy().ravel(); ue = AMP * np.cos(w * tn)
    return tn, ue, up, error_l2(up, ue)

ks = [2, 4, 6, 8, 10] if not RAPIDO else [2, 8]
err_tanh, err_siren, sol8 = [], [], None
print(f"{'ω':>6} {'tanh':>12} {'seno (SIREN)':>14}")
for kk in ks:
    w = kk * np.pi
    fijar_semilla(0); tn, ue, ut, et = entrenar_osc(RedTanh(), w)
    fijar_semilla(0); _, _, us, es = entrenar_osc(SIREN(), w)
    err_tanh.append(et); err_siren.append(es)
    print(f"{kk:>4}π {et:>12.3e} {es:>14.3e}")
    if kk == 8: sol8 = dict(tn=tn, ue=ue, tanh=ut, siren=us, et=et, es=es)

torch.set_default_dtype(torch.float32)        # se restablece la precisión simple

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(sol8["tn"], sol8["ue"], color="#222", lw=2.4, label="Exacta 5cos(8πt)")
ax[0].plot(sol8["tn"], sol8["tanh"], "--", color=ROJO, lw=1.8, label=f"tanh (L2={sol8['et']:.1e})")
ax[0].plot(sol8["tn"], sol8["siren"], "-.", color=AZUL, lw=1.8, label=f"seno/SIREN (L2={sol8['es']:.1e})")
ax[0].set_title("(a) ω = 8π"); ax[0].set_xlabel("t"); ax[0].set_ylabel("u(t)"); ax[0].legend(fontsize=9)
ax[1].semilogy(ks, err_tanh, "o--", color=ROJO, label="tanh")
ax[1].semilogy(ks, err_siren, "s-.", color=AZUL, label="seno/SIREN")
ax[1].set_title("(b) Error L2 relativo frente a ω = kπ"); ax[1].set_xlabel("k"); ax[1].legend()
fig.tight_layout(); plt.show()

## 5. Problemas directos

### 5.1 Ecuación logística

**Problema.** $N(t)$ es la cantidad de supermercados del país que han adoptado un sistema de revisión computarizado. Al inicio solo uno lo usa y la adopción se satura en 2000. ¿Cuántos lo habrán adoptado en $t=10$?

$$\frac{dN}{dt}=N\,(1-0{,}0005\,N),\qquad N(0)=1,\qquad t\in[0,10]\qquad(r=1,\ K=2000).$$

**Solución analítica.** Separando variables y por fracciones parciales,
$$\left(\frac1N+\frac1{2000-N}\right)dN=dt\;\Rightarrow\;\frac{N}{2000-N}=C_1e^{t},\quad C_1=\frac1{1999}\;\Rightarrow\;N(t)=\frac{2000}{1+1999\,e^{-t}}.$$

**PINN.** Con el cambio $w=\ln N$ y la restricción dura $\hat w_\theta=(1-e^{-t})N_\theta(t)$, que garantiza $\hat N_\theta(0)=1$:
$$R=\hat w_\theta'-\Big(1-\frac{e^{\hat w_\theta}}{K}\Big),\qquad \mathcal{L}=\frac1{400}\sum_{i=1}^{400}R(t_i)^2.$$

**Arquitectura:** $1-32-32-32-1$ · $\tanh$ · Adam $15\,000$ + L-BFGS.

In [ ]:
r_log, K_log, T_log = 1.0, 2000.0, 10.0
t_log = np.linspace(0, T_log, 500)
N_analitica = K_log / (1 + (K_log - 1) * np.exp(-r_log * t_log))

dde.config.set_random_seed(1234)
geom = dde.geometry.TimeDomain(0, T_log)
def ode_log(t, w):                                    # residuo en la variable w = ln N
    return dde.grad.jacobian(w, t) - (1 - dde.backend.exp(w) / K_log)

data = dde.data.PDE(geom, ode_log, [], num_domain=400, num_test=200)
net = dde.nn.FNN([1] + [32] * 3 + [1], "tanh", "Glorot uniform")
net.apply_output_transform(lambda t, y: (1 - dde.backend.exp(-t)) * y)   # restricción dura
modelo_log = dde.Model(data, net)
modelo_log.compile("adam", lr=1e-3)
modelo_log.train(iterations=IT(15000), display_every=5000)
modelo_log.compile("L-BFGS")
modelo_log.train()

N_pinn = np.exp(modelo_log.predict(t_log.reshape(-1, 1)).ravel())   # N = e^w
print(f"\nPINN: error L2 relativo = {error_l2(N_pinn, N_analitica):.3e}")
print(f"N(10): exacta = {N_analitica[-1]:.2f}   PINN = {N_pinn[-1]:.2f}  (≈ {round(N_pinn[-1])} supermercados)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(t_log, N_analitica, color="black", lw=2.2)
ax[0].text(0.05, 0.8, r"$N(t)=\dfrac{2000}{1+1999\,e^{-t}}$", transform=ax[0].transAxes, fontsize=12,
           bbox=dict(boxstyle="round", facecolor="white", edgecolor=GRIS))
ax[0].set_title("(a) Solución analítica")
ax[1].plot(t_log, N_analitica, color=GRIS, lw=3, alpha=0.5, label="Analítica")
ax[1].plot(t_log, N_pinn, "-.", color=AZUL, lw=1.8, label="PINN")
ax[1].set_title("(c) Solución con PINN"); ax[1].legend()
for a in ax: a.set_xlabel("t"); a.set_ylabel("N(t)")
fig.tight_layout(); plt.show()

print(f"{'t':>4} {'Exacta':>12} {'PINN':>12}")
for tv in [4, 5, 7, 9, 10]:
    ne = K_log / (1 + (K_log - 1) * np.exp(-tv))
    npinn = np.exp(modelo_log.predict(np.array([[float(tv)]]))[0, 0])
    print(f"{tv:>4} {ne:>12.3f} {npinn:>12.3f}")

### 5.2 Oscilador armónico simple

$$y''+4y=0,\qquad y(0)=1,\quad y'(0)=0,\qquad t\in[0,2\pi].$$

**Solución analítica.** $r^2+4=0\Rightarrow r=\pm2i$, $\;y=C_1\cos2t+C_2\,\mathrm{sen}\,2t$; con $C_1=1$ y $2C_2=0$: $\;y(t)=\cos2t$.

**PINN.** $\;R=\hat y_\theta''+\omega^2\hat y_\theta,\qquad \mathcal{L}=\dfrac1{200}\displaystyle\sum_{i=1}^{200}R(t_i)^2+\big(\hat y_\theta(0)-1\big)^2+\big(\hat y_\theta'(0)\big)^2.$

**Como sistema** (red de dos salidas): $u_1'=u_2$, $u_2'=-\omega^2u_1$.

**Arquitectura:** $1-32-32-32-1$ · $\tanh$ · Adam $15\,000$ + L-BFGS · 200 puntos de colocación.

In [ ]:
w_os, T_os = 2.0, 2 * np.pi
t_os = np.linspace(0, T_os, 500).reshape(-1, 1)
y_exacta_os = np.cos(w_os * t_os)
geom = dde.geometry.TimeDomain(0, T_os)

# --- formulación directa (segundo orden) ---
dde.config.set_random_seed(1234)
def ode_os(t, y):
    return dde.grad.hessian(y, t) + w_os ** 2 * y
ic1 = dde.icbc.IC(geom, lambda x: 1.0, lambda _, on: on)                       # y(0) = 1
ic2 = dde.icbc.OperatorBC(geom, lambda x, y, X: dde.grad.jacobian(y, x),       # y'(0) = 0
                          lambda x, on: on and np.isclose(x[0], 0))
data = dde.data.PDE(geom, ode_os, [ic1, ic2], num_domain=200, num_boundary=2, num_test=200)
modelo_directo = dde.Model(data, dde.nn.FNN([1] + [32] * 3 + [1], "tanh", "Glorot uniform"))
modelo_directo.compile("adam", lr=1e-3); modelo_directo.train(iterations=IT(15000), display_every=5000)
modelo_directo.compile("L-BFGS"); modelo_directo.train()
y_directo = modelo_directo.predict(t_os).ravel()

# --- sistema de primer orden (red de dos salidas) ---
dde.config.set_random_seed(1234)
def sistema_os(t, u):
    du1 = dde.grad.jacobian(u, t, i=0); du2 = dde.grad.jacobian(u, t, i=1)
    return [du1 - u[:, 1:2], du2 + w_os ** 2 * u[:, 0:1]]
ic_u1 = dde.icbc.IC(geom, lambda x: 1.0, lambda _, on: on, component=0)
ic_u2 = dde.icbc.IC(geom, lambda x: 0.0, lambda _, on: on, component=1)
data = dde.data.PDE(geom, sistema_os, [ic_u1, ic_u2], num_domain=200, num_boundary=2, num_test=200)
modelo_sistema = dde.Model(data, dde.nn.FNN([1] + [32] * 3 + [2], "tanh", "Glorot uniform"))
modelo_sistema.compile("adam", lr=1e-3); modelo_sistema.train(iterations=IT(15000), display_every=5000)
modelo_sistema.compile("L-BFGS"); modelo_sistema.train()
y_sistema = modelo_sistema.predict(t_os)[:, 0]

print(f"\nFormulación directa: L2 = {error_l2(y_directo, y_exacta_os):.3e}")
print(f"Sistema 1.er orden:  L2 = {error_l2(y_sistema, y_exacta_os):.3e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(t_os, y_exacta_os, color=GRIS, lw=3, alpha=0.6, label="Exacta cos(2t)")
ax[0].plot(t_os, y_directo, "--", color=AZUL, lw=1.8, label="PINN (2.º orden)")
ax[0].plot(t_os, y_sistema, ":", color=VERDE, lw=2, label="PINN (sistema)")
ax[0].set_title("Ecuación frente a sistema"); ax[0].set_xlabel("t"); ax[0].set_ylabel("y(t)"); ax[0].legend(fontsize=9)
ax[1].semilogy(t_os, np.abs(y_directo - y_exacta_os.ravel()) + 1e-16, color=AZUL, label="2.º orden")
ax[1].semilogy(t_os, np.abs(y_sistema - y_exacta_os.ravel()) + 1e-16, color=VERDE, label="sistema")
ax[1].set_title("Error absoluto (escala log)"); ax[1].set_xlabel("t"); ax[1].legend()
fig.tight_layout(); plt.show()

print(f"{'t':>7} {'Exacta':>11} {'PINN':>11}")
for tv in [1.257, 3.142, 6.283]:
    print(f"{tv:>7} {np.cos(2 * tv):>11.6f} {modelo_directo.predict(np.array([[tv]]))[0, 0]:>11.6f}")

## 6. Problemas inversos

$$u'=f(t,u;\lambda),\qquad \lambda\ \text{desconocido},\qquad \text{datos }\{(t_j,u_j)\}_{j=1}^{N_d}$$

$$\mathcal{L}(\theta,\lambda)=\frac1{N_c}\sum_{i=1}^{N_c}\big|u_\theta'(t_i)-f(t_i,u_\theta;\lambda)\big|^2+\frac1{N_d}\sum_{j=1}^{N_d}\big|u_\theta(t_j)-u_j\big|^2$$

$\lambda$ es **un parámetro entrenable más**: se optimiza junto con los pesos $\theta$, y un solo entrenamiento reconstruye $u(t)$ y estima $\lambda$.

### 6.1 Propagación de una gripe

Campus de 1000 estudiantes, $k$ desconocido (problema tomado de Zill, 2009):
$$x'=k\,x(1000-x),\quad x(0)=1.\qquad\text{Con } r=1000k:\quad x(t)=\frac{1000}{1+999\,e^{-rt}}.$$
De $x(4)=50$: $\;r=\tfrac14\ln\tfrac{999}{19}=0{,}9906$ (referencia). La PINN parte de $r=0{,}5$ y usa 7 observaciones.

**Arquitectura:** $1-32-32-32-1$ · $\tanh$ · Adam $15\,000$ + L-BFGS.

In [ ]:
t_gr = np.array([4, 5, 6, 7, 8, 9, 10.]).reshape(-1, 1)
x_gr = np.array([50, 124, 276, 507, 735, 882, 953.]).reshape(-1, 1)
K_gr, T_gr = 1000.0, 10.0
r_real = 0.25 * np.log(999 / 19)

dde.config.set_random_seed(1234)
r_var = dde.Variable(0.5)                                   # parámetro desconocido
geom = dde.geometry.TimeDomain(0, T_gr)
def ode_gr(t, w):                                           # residuo en w = ln x
    return dde.grad.jacobian(w, t) - r_var * (1 - dde.backend.exp(w) / K_gr)
observe = dde.icbc.PointSetBC(t_gr, np.log(x_gr), component=0)
data = dde.data.PDE(geom, ode_gr, [observe], num_domain=200, anchors=t_gr)
net = dde.nn.FNN([1] + [32] * 3 + [1], "tanh", "Glorot uniform")
net.apply_output_transform(lambda t, y: (1 - dde.backend.exp(-t)) * y)   # w(0) = 0, es decir x(0) = 1
modelo_gr = dde.Model(data, net)
seg = dde.callbacks.VariableValue(r_var, period=max(IT(15000) // 15, 1), filename="r_evolucion.dat")
modelo_gr.compile("adam", lr=1e-3, external_trainable_variables=[r_var])
modelo_gr.train(iterations=IT(15000), display_every=5000, callbacks=[seg])
modelo_gr.compile("L-BFGS", external_trainable_variables=[r_var])
modelo_gr.train()

r_est = leer(r_var)
x6 = np.exp(modelo_gr.predict(np.array([[6.0]]))[0, 0])
print(f"\nr estimado = {r_est:.4f}   referencia = {r_real:.4f}   error = {100 * abs(r_est - r_real) / r_real:.3f} %")
print(f"Predicción x(6) = {x6:.1f}   (dato: 276)")

In [ ]:
pasos, valores = [], []
with open("r_evolucion.dat") as f:
    for linea in f:
        p = linea.split()
        if len(p) >= 2:
            pasos.append(int(p[0])); valores.append(float(p[1].strip("[]")))
tt = np.linspace(0, T_gr, 500)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(pasos, valores, "o-", color=AZUL, ms=3, label="r estimado")
ax[0].axhline(r_real, ls="--", color=ROJO, label=f"referencia {r_real:.4f}")
ax[0].set_xlabel("Iteración"); ax[0].set_ylabel("r"); ax[0].set_title("Convergencia del parámetro"); ax[0].legend()
ax[1].plot(tt, K_gr / (1 + 999 * np.exp(-r_real * tt)), color=GRIS, lw=3, alpha=0.5, label="Referencia")
ax[1].plot(tt, np.exp(modelo_gr.predict(tt.reshape(-1, 1)).ravel()), "--", color=AZUL, label="PINN")
ax[1].scatter(t_gr, x_gr, color=ROJO, zorder=5, label="Datos")
ax[1].set_xlabel("t (días)"); ax[1].set_ylabel("infectados"); ax[1].set_title("Reconstrucción de la epidemia"); ax[1].legend()
fig.tight_layout(); plt.show()

### 6.2 Oscilador con datos de un simulador (AulaQuest)

Datos exportados del simulador masa-resorte de AulaQuest (https://aulaquest.com/s/fisica/masa-resorte/index.php), configurado con $m=1$ kg, $k=20$ N/m y $b=0{,}8$ kg/s:

$$m\,y''+b\,y'+k\,y=0,\qquad r^2+0{,}8\,r+20=0\;\Rightarrow\;r=-0{,}4\pm4{,}454\,i,$$
$$\gamma=\frac{b}{2m}=0{,}4\ \text{s}^{-1},\qquad \omega_0=\sqrt{k/m}=4{,}472,\qquad \omega_d=\sqrt{\omega_0^2-\gamma^2}=4{,}454\ \text{rad/s}.$$

**Incógnitas:** $b$ y $k$ a partir solo de las 27 posiciones. Se centra la señal en el equilibrio y se normaliza la amplitud.

**Arquitectura:** $1-64-64-64-1$ · $\tanh$ · Adam $25\,000$ (dos fases) + L-BFGS.

In [ ]:
# 27 observaciones (t en s, y en m) exportadas del simulador
AULA = [
    [0.0, 0.299405],
    [0.783333, 1.495821],
    [1.133333, 0.896735],
    [1.35, 0.614379],
    [1.55, 0.66117],
    [1.733333, 0.9043],
    [1.95, 1.20452],
    [2.133333, 1.296814],
    [2.316667, 1.200037],
    [2.483333, 1.015849],
    [2.65, 0.844379],
    [2.85, 0.768151],
    [3.0, 0.823765],
    [3.166667, 0.957392],
    [3.35, 1.104039],
    [3.533333, 1.167216],
    [3.716667, 1.121189],
    [3.9, 1.007686],
    [4.116667, 0.88946],
    [4.3, 0.86832],
    [4.483333, 0.924498],
    [4.683333, 1.022442],
    [4.866667, 1.084674],
    [5.083333, 1.079905],
    [5.266667, 1.021843],
    [5.45, 0.956139],
    [5.75, 0.925814]
]
datos_aula = np.array(AULA)
t_obs, y_obs = datos_aula[:, :1], datos_aula[:, 1:]
T_aula = float(t_obs.max())

y_eq = y_obs.mean()                      # equilibrio
u_scale = np.abs(y_obs - y_eq).max()     # escala de amplitud
u_norm = (y_obs - y_eq) / u_scale

dde.config.set_random_seed(1234)
b_var, k_var = dde.Variable(0.5), dde.Variable(15.0)     # valores reales: 0,8 y 20
geom = dde.geometry.TimeDomain(0, T_aula)
def res_aula(t, u):
    return 1.0 * dde.grad.hessian(u, t) + b_var * dde.grad.jacobian(u, t) + k_var * u
observe = dde.icbc.PointSetBC(t_obs, u_norm, component=0)
data = dde.data.PDE(geom, res_aula, [observe], num_domain=500, anchors=t_obs)
modelo_aula = dde.Model(data, dde.nn.FNN([1] + [64] * 3 + [1], "tanh", "Glorot uniform"))
ext = [b_var, k_var]
modelo_aula.compile("adam", lr=1e-3, external_trainable_variables=ext, loss_weights=[0.1, 100])   # fase 1
modelo_aula.train(iterations=IT(10000), display_every=5000)
modelo_aula.compile("adam", lr=5e-4, external_trainable_variables=ext, loss_weights=[1, 100])     # fase 2
modelo_aula.train(iterations=IT(15000), display_every=5000)
modelo_aula.compile("L-BFGS", external_trainable_variables=ext, loss_weights=[1, 100])
modelo_aula.train()

b_est, k_est = leer(b_var), leer(k_var)
print(f"\n{'':10} {'Real':>8} {'PINN':>10} {'Error':>9}")
print(f"{'b (kg/s)':10} {0.8:>8.4f} {b_est:>10.4f} {100 * abs(b_est - 0.8) / 0.8:>8.2f} %")
print(f"{'k (N/m)':10} {20.0:>8.4f} {k_est:>10.4f} {100 * abs(k_est - 20) / 20:>8.2f} %")

tt = np.linspace(0, T_aula, 400).reshape(-1, 1)
plt.figure(figsize=(10, 4))
plt.plot(t_obs, y_obs, "o", color=ROJO, ms=6, label="Observaciones (simulador)", zorder=5)
plt.plot(tt, modelo_aula.predict(tt) * u_scale + y_eq, color=AZUL, lw=2, label="PINN (parámetros estimados)")
plt.xlabel("t (s)"); plt.ylabel("y (m)"); plt.title("Oscilador con datos del simulador: PINN frente a observaciones")
plt.legend(); plt.show()

### 6.3 Sistema de Lorenz (régimen no caótico)

$$x'=\sigma(y-x),\qquad y'=x(\rho-z)-y,\qquad z'=xy-\beta z,\qquad (x_0,y_0,z_0)=(-8,7,27).$$

Datos: 25 observaciones en $t\in[0,3]$ con $\sigma=10$, $\rho=15$, $\beta=8/3$ (conjunto `Lorenz.npz` del repositorio de DeepXDE). Los tres parámetros parten de 1.

**Arquitectura:** $1-40-40-40-3$ · $\tanh$ · Adam $20\,000$ + L-BFGS.

In [ ]:
# 25 observaciones: t, x, y, z (régimen no caótico, rho = 15)
L15 = [
    [0.0, -8.0, 7.0, 27.0],
    [0.0107173821077742, -6.431736688846125, 7.797594411616799, 25.675056426006645],
    [0.0643042926466452, -0.1825283848827821, 8.791526010944368, 20.99658491786237],
    [0.1226243956746697, 3.572291345116445, 7.808073500654382, 18.845378862549893],
    [0.2038973829529623, 5.36627000030819, 6.049928574988049, 17.53854034426507],
    [0.2774039015171674, 5.351484519233177, 4.862868847201283, 16.370385214791437],
    [0.3660593203757163, 4.804107663983576, 4.212047509467882, 14.71050379668103],
    [0.4648463623733734, 4.436778471612032, 4.328404183905912, 12.9758144243041],
    [0.5774702337008112, 4.674796934076926, 5.194091954008366, 11.683386720189198],
    [0.7441094050300586, 6.106115575515522, 7.171871005185811, 12.032343139632296],
    [0.8781570930284631, 7.31176996097192, 7.847942492447129, 14.319342882713938],
    [1.0122047810268675, 7.195168414224626, 6.50968884620264, 16.071570028997733],
    [1.1500468631774832, 5.884151703985276, 4.908537988094659, 15.33987160763446],
    [1.2828075948244018, 4.980455914306564, 4.660057939616842, 13.551726744699966],
    [1.4251076467568653, 5.079791143522666, 5.503792076990994, 12.222599479768256],
    [1.5817275555316088, 6.139173442744376, 6.948718901303769, 12.567187916176062],
    [1.7539737656401888, 7.126206862496815, 7.254853046338332, 14.78184718947654],
    [1.8976423335247845, 6.642554437869463, 5.946013415021821, 15.58157268090812],
    [2.049233346104226, 5.563054546028784, 5.007054838413085, 14.43028766544123],
    [2.1879595463534405, 5.200874122149641, 5.247387413198745, 13.075851511049471],
    [2.376502181363779, 5.916202288622185, 6.513742935737766, 12.726090338247136],
    [2.537477369055736, 6.773035396203111, 7.093680594403205, 14.11026310807365],
    [2.699223679350244, 6.66760163260164, 6.2464222392351605, 15.19672981734528],
    [2.8642406091651025, 5.7951836573319975, 5.321187488504717, 14.491484767243996],
    [3.0, 5.430238438217022, 5.391742630070061, 13.438345631864737]
]

def inverso_lorenz(datos, semilla, archivo):
    # problema inverso de Lorenz: devuelve el modelo, los parámetros y su historial
    datos = np.array(datos)
    t_o, u_o = datos[:, :1], datos[:, 1:]
    x0, y0, z0 = u_o[0]
    C1, C2, C3 = dde.Variable(1.0), dde.Variable(1.0), dde.Variable(1.0)
    dde.config.set_random_seed(semilla)
    geom = dde.geometry.TimeDomain(0, float(t_o.max()))
    def lorenz(t, u):
        x, y, z = u[:, 0:1], u[:, 1:2], u[:, 2:3]
        return [dde.grad.jacobian(u, t, i=0) - C1 * (y - x),
                dde.grad.jacobian(u, t, i=1) - x * (C2 - z) + y,
                dde.grad.jacobian(u, t, i=2) - x * y + C3 * z]
    ini = lambda _, on: on
    constante = lambda v: (lambda X: v)
    bcs = [dde.icbc.IC(geom, constante(v), ini, component=c) for c, v in enumerate([x0, y0, z0])]
    bcs += [dde.icbc.PointSetBC(t_o, u_o[:, c:c + 1], component=c) for c in range(3)]
    data = dde.data.PDE(geom, lorenz, bcs, num_domain=400, num_boundary=2, anchors=t_o)
    modelo = dde.Model(data, dde.nn.FNN([1] + [40] * 3 + [3], "tanh", "Glorot uniform"))
    var = dde.callbacks.VariableValue([C1, C2, C3], period=max(IT(20000) // 30, 1), filename=archivo)
    modelo.compile("adam", lr=1e-3, external_trainable_variables=[C1, C2, C3])
    modelo.train(iterations=IT(20000), display_every=5000, callbacks=[var])
    modelo.compile("L-BFGS", external_trainable_variables=[C1, C2, C3])
    modelo.train(callbacks=[var])
    return modelo, t_o, u_o, [leer(C1), leer(C2), leer(C3)]

def historial(archivo):
    its, vals = [], []
    with open(archivo) as f:
        for linea in f:
            it, arr = linea.strip().split(" ", 1)
            its.append(int(it)); vals.append([float(v) for v in re.findall(r"[-+]?\d*\.?\d+(?:e[-+]?\d+)?", arr)])
    return np.array(its), np.array(vals)

def tabla_lorenz(est, rho):
    print(f"\n{'':6} {'Real':>9} {'PINN':>9} {'Error':>9}")
    for n, real, e in zip(["σ", "ρ", "β"], [10.0, rho, 8 / 3], est):
        print(f"{n:6} {real:>9.4f} {e:>9.4f} {100 * abs(e - real) / real:>8.2f} %")

modelo_l15, t_l15, u_l15, est15 = inverso_lorenz(L15, 42, "lorenz15.dat")
tabla_lorenz(est15, 15.0)

its, vals = historial("lorenz15.dat")
plt.figure(figsize=(8, 4))
for j, (n, real) in enumerate([("σ", 10), ("ρ", 15), ("β", 8 / 3)]):
    plt.plot(its, vals[:, j], label=n); plt.axhline(real, ls="--", color=f"C{j}", alpha=0.5)
plt.xlabel("Iteración"); plt.ylabel("Valor"); plt.title("Convergencia de los parámetros de Lorenz"); plt.legend(); plt.show()

## 7. Comparación, límites y conclusiones

### 7.1 Donde la PINN gana (1): problema rígido

$$y'=\cos t-50\,(y-\mathrm{sen}\,t),\qquad y(0)=1\quad\Rightarrow\quad y(t)=\mathrm{sen}\,t+e^{-50t},\qquad t\in[0,3].$$

RK4 explícito es estable solo si $h\alpha\le2{,}785$. Con $h=0{,}1$, $h\alpha=5$ y la solución diverge; la PINN no depende de un paso de malla.

**Arquitectura de la PINN:** $1-64-64-64-64-1$ · $\tanh$ · Adam $20\,000$ + L-BFGS · 300 puntos de anclaje en $[0;\,0{,}3]$ · restricción dura.

In [ ]:
a_r, T_r = 50.0, 3.0
f_r = lambda t, y: np.cos(t) - a_r * (y - np.sin(t))
exacta_r = lambda t: np.sin(t) + np.exp(-a_r * t)

def rk4(f, t0, y0, T, h):
    n = int(round((T - t0) / h)); tt = np.linspace(t0, t0 + n * h, n + 1)
    y = np.zeros(n + 1); y[0] = y0
    for i in range(n):
        k1 = f(tt[i], y[i]); k2 = f(tt[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(tt[i] + h / 2, y[i] + h / 2 * k2); k4 = f(tt[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return tt, y

t1, y1 = rk4(f_r, 0, 1.0, T_r, 0.1)
t2, y2 = rk4(f_r, 0, 1.0, T_r, 0.05)
print(f"RK4, h = 0,1  (hα = 5):   |y(3)| = {abs(y1[-1]):.2e}  -> diverge")
print(f"RK4, h = 0,05 (hα = 2,5): error máx = {np.max(np.abs(y2 - exacta_r(t2))):.3f}  -> estable")

te = np.linspace(0, T_r, 2000)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].semilogy(t1, np.abs(y1), "o-", color=ROJO, ms=4, label="RK4, h = 0,1")
ax[0].semilogy(te, np.abs(exacta_r(te)), color="#555", lw=2, label="Exacta")
ax[0].set_xlabel("t"); ax[0].set_ylabel("|y(t)| (escala log)"); ax[0].set_title("(a) RK4 explícito diverge"); ax[0].legend()
ax[1].plot(te, exacta_r(te), color="#555", lw=2.5, label="Exacta")
ax[1].plot(t2, y2, "s--", color=AZUL, ms=4, label="RK4, h = 0,05")
ax[1].set_xlim(0, 0.6); ax[1].set_xlabel("t"); ax[1].set_title("(b) Estable, pero no resuelve el transitorio"); ax[1].legend()
fig.tight_layout(); plt.show()

In [ ]:
dde.config.set_random_seed(1234)
geom = dde.geometry.TimeDomain(0, T_r)
def ode_r(t, y):
    return dde.grad.jacobian(y, t) - (dde.backend.cos(t) - a_r * (y - dde.backend.sin(t)))
anclaje = np.linspace(0, 0.3, 300).reshape(-1, 1)                    # puntos donde ocurre el transitorio
data = dde.data.PDE(geom, ode_r, [], num_domain=400, num_test=500, anchors=anclaje)
net = dde.nn.FNN([1] + [64] * 4 + [1], "tanh", "Glorot uniform")
net.apply_output_transform(lambda t, y: 1.0 + (1 - dde.backend.exp(-t)) * y)   # y(0) = 1 exacto
modelo_r = dde.Model(data, net)
modelo_r.compile("adam", lr=1e-3); modelo_r.train(iterations=IT(20000), display_every=5000)
modelo_r.compile("L-BFGS"); modelo_r.train()

t_r = np.linspace(0, T_r, 1000)
y_r = modelo_r.predict(t_r.reshape(-1, 1)).ravel()
print(f"\nPINN: error L2 relativo = {error_l2(y_r, exacta_r(t_r)):.2e}  (estable en todo el dominio)")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, m in zip(ax, [t_r <= 3, t_r <= 0.3]):
    a.plot(t_r[m], exacta_r(t_r[m]), color=GRIS, lw=3, alpha=0.6, label="Exacta")
    a.plot(t_r[m], y_r[m], "-.", color=AZUL, lw=1.6, label="PINN")
    a.set_xlabel("t"); a.legend()
ax[0].set_title("(a) Dominio completo"); ax[1].set_title("(b) Transitorio [0; 0,3]")
fig.tight_layout(); plt.show()

### 7.2 Donde la PINN gana (2): problema de contorno

$$y''+y=0,\qquad y(0)=0,\qquad y(\pi/2)=1\qquad\Rightarrow\qquad y(t)=\mathrm{sen}\,t.$$

Runge-Kutta necesita el **método de disparo**: integrar el PVI con $y'(0)=s$ y buscar $s$ tal que $y(\pi/2;\,s)=1$, lo que exige varias integraciones. La PINN impone ambas condiciones en la pérdida:

$$\mathcal{L}=\frac1{N_c}\sum_i R(t_i)^2+\hat y_\theta(0)^2+\big(\hat y_\theta(\tfrac{\pi}{2})-1\big)^2.$$

**Arquitectura:** $1-32-32-32-1$ · $\tanh$ · Adam $12\,000$ + L-BFGS · pesos $1:100:100$.

In [ ]:
SEMILLA = 1234
T_J = np.pi / 2
geom = dde.geometry.TimeDomain(0, T_J)
def res_J(t, y):
    return dde.grad.hessian(y, t) + y
bc_izq = dde.icbc.DirichletBC(geom, lambda x: 0.0, lambda x, on: on and np.isclose(x[0], 0))
bc_der = dde.icbc.DirichletBC(geom, lambda x: 1.0, lambda x, on: on and np.isclose(x[0], T_J))

dde.config.set_random_seed(SEMILLA)
data = dde.data.PDE(geom, res_J, [bc_izq, bc_der], num_domain=400, num_boundary=2, num_test=1000)
modelo_J = dde.Model(data, dde.nn.FNN([1] + [32] * 3 + [1], "tanh", "Glorot uniform"))
modelo_J.compile("adam", lr=1e-3, loss_weights=[1, 100, 100]); modelo_J.train(iterations=IT(12000), display_every=4000)
modelo_J.compile("L-BFGS", loss_weights=[1, 100, 100]); modelo_J.train()

t_J = np.linspace(0, T_J, 1000).reshape(-1, 1)
y_J = modelo_J.predict(t_J)
print(f"\nProblema de contorno: error L2 relativo = {error_l2(y_J, np.sin(t_J)):.2e}")

plt.figure(figsize=(7, 3.8))
plt.plot(t_J, np.sin(t_J), color=GRIS, lw=3, alpha=0.6, label="Exacta sen t")
plt.plot(t_J, y_J, "--", color=AZUL, lw=1.8, label="PINN")
plt.scatter([0, T_J], [0, 1], color=ROJO, zorder=5, label="Condiciones de contorno")
plt.xlabel("t"); plt.ylabel("y(t)"); plt.title("Problema de contorno sin método de disparo"); plt.legend(); plt.show()

### 7.3 Límites del método

**Lorenz caótico** ($\rho=28$): misma arquitectura y procedimiento que en el régimen no caótico ($1-40-40-40-3$ · $\tanh$); solo cambian los datos. La sensibilidad a las condiciones iniciales vuelve irregular la pérdida y los parámetros no se recuperan.

In [ ]:
# 25 observaciones: t, x, y, z (régimen caótico, rho = 28)
L28 = [
    [0.0, -8.0, 7.0, 27.0],
    [0.125, 1.8163855497832515, 5.595386363043799, 18.1595691720145],
    [0.25, 6.179064671823058, 10.306527509276744, 16.46343526951299],
    [0.375, 12.37852352672277, 16.8965390274682, 26.49542429202508],
    [0.5, 11.83331427668488, 6.150497643906148, 36.69558326153254],
    [0.625, 4.482247161250923, 0.3384220793542585, 28.23274267755604],
    [0.75, 1.8998528961311605, 1.4084083360152104, 20.431799181842603],
    [0.875, 2.4356747224965534, 3.720435283517889, 15.180171474197564],
    [1.0, 5.523676645459319, 9.518798965807273, 13.583499720967003],
    [1.125, 12.572640443904952, 18.65799666723814, 24.375144449336403],
    [1.25, 12.929021825667514, 6.126257367181252, 38.88240856676408],
    [1.375, 3.745417655621864, -1.441854580862918, 28.907604898563736],
    [1.5, 0.2753973631217814, -0.8786073988607225, 20.508232511995164],
    [1.625, -0.5727453601456642, -1.0942515220827285, 14.718438440117689],
    [1.75, -1.557965281163708, -2.817567505728934, 10.758308121998851],
    [1.875, -4.526310202211399, -8.519789299984584, 9.45815884735953],
    [2.0, -12.59871732626702, -20.782637965569727, 20.84219249183192],
    [2.125, -14.484256217000572, -6.053581190887099, 42.0247026389812],
    [2.25, -2.583418373343115, 4.038222624624551, 29.81251517126852],
    [2.375, 2.03485900696851, 4.010172469887376, 21.491548503967618],
    [2.5, 4.521302961964662, 7.062997026965869, 17.31033989958242],
    [2.625, 9.181288290258442, 14.010924006692315, 20.226171416034987],
    [2.75, 13.453060270113443, 13.141731783495814, 33.90096886978149],
    [2.875, 7.999227111483591, 2.1659561681970274, 32.753355640019514],
    [3.0, 3.1552708998264283, 1.362420582228825, 24.156265153765617]
]

modelo_l28, t_l28, u_l28, est28 = inverso_lorenz(L28, 1234, "lorenz28.dat")
tabla_lorenz(est28, 28.0)
print("\nEn régimen caótico la estimación no converge: el problema inverso se vuelve mal planteado.")

tp = np.linspace(0, float(t_l28.max()), 600).reshape(-1, 1)
Yp = modelo_l28.predict(tp)
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for j, c in enumerate(["x", "y", "z"]):
    axs[j].plot(tp, Yp[:, j], color=AZUL, label="PINN")
    axs[j].plot(t_l28, u_l28[:, j], ".", color=ROJO, ms=8, label="Observaciones")
    axs[j].set_ylabel(c)
axs[0].legend(loc="upper right"); axs[-1].set_xlabel("t"); fig.suptitle("Lorenz caótico (ρ = 28)"); fig.tight_layout(); plt.show()

**Oscilador de Van der Pol:** $\;y''-\mu(1-y^2)\,y'+y=0,\quad y(0)=2,\ y'(0)=0$, en $t\in[0,10]$. Es un oscilador de relajación con transiciones rápidas; la referencia se calcula con `solve_ivp` (tolerancias estrictas).

**Arquitectura:** $1-64-64-64-1$ · $\tanh$ · Adam + L-BFGS, con la restricción dura $\hat y_\theta=2+t^2N_\theta(t)$.

> **Nota:** el valor de $\mu$ controla la rigidez del oscilador; aquí se usa `MU = 1.0`. Con valores mayores (por ejemplo $\mu=5$) las transiciones son más abruptas y el problema es más difícil.

In [ ]:
MU = 1.0
T_v = 10.0
f_vdp = lambda t, u: [u[1], MU * (1 - u[0] ** 2) * u[1] - u[0]]
t_v = np.linspace(0, T_v, 1000)
y_ref_v = solve_ivp(f_vdp, (0, T_v), [2, 0], t_eval=t_v, rtol=1e-11, atol=1e-12).y[0]

dde.config.set_random_seed(SEMILLA)
geom = dde.geometry.TimeDomain(0, T_v)
def res_vdp(t, y):
    return dde.grad.hessian(y, t) - MU * (1 - y ** 2) * dde.grad.jacobian(y, t) + y
data = dde.data.PDE(geom, res_vdp, [], num_domain=400, num_test=1000)
net = dde.nn.FNN([1] + [64] * 3 + [1], "tanh", "Glorot uniform")
net.apply_output_transform(lambda t, y: 2.0 + t ** 2 * y)            # y(0) = 2, y'(0) = 0
modelo_v = dde.Model(data, net)
modelo_v.compile("adam", lr=1e-3); modelo_v.train(iterations=IT(55000), display_every=10000)
modelo_v.compile("L-BFGS"); modelo_v.train()

y_v = modelo_v.predict(t_v.reshape(-1, 1)).ravel()
print(f"\nVan der Pol: error L2 relativo = {error_l2(y_v, y_ref_v):.2e}")

plt.figure(figsize=(8, 3.8))
plt.plot(t_v, y_ref_v, color="k", lw=2.4, alpha=0.5, label="Referencia")
plt.plot(t_v, y_v, "--", color=ROJO, lw=1.8, label="PINN")
plt.xlabel("t"); plt.ylabel("y(t)"); plt.title("Oscilador de Van der Pol"); plt.legend(); plt.show()

### 7.4 Ventajas de las PINNs y conclusiones

**Lo que aporta**
* **Problemas inversos:** solución y parámetros en un solo entrenamiento.
* **Datos escasos:** la física suple las observaciones ausentes.
* **Problemas rígidos:** sin restricción de paso.
* **Contorno:** las condiciones entran en la pérdida, sin disparo.
* **Solución continua y diferenciable** en todo el dominio.
* **Se extiende a EDP** con el mismo esquema (ver `PINNs_EDP.ipynb`).

**Lo que exige**
* Activación acorde con la solución ($\tanh$ o seno).
* Entrenamiento y ajuste de hiperparámetros.
* Cuidado con el caos y las múltiples escalas.

**Conclusiones**
1. Con idéntica arquitectura y entrenamiento, la PINN y la red estándar coinciden donde hay datos; donde no los hay, la PINN extrapola: $\mathcal{L}_F$ actúa como regularizador físico.
2. La activación debe ser suave y acorde con la solución: $\tanh$ para soluciones suaves y seno (SIREN) para oscilaciones rápidas.
3. La PINN es estable en el problema rígido, donde RK4 explícito diverge, y resuelve el problema de contorno sin disparo.
4. En problemas inversos identifica parámetros con pocas observaciones: Lorenz no caótico con errores menores al 0,5 % y un oscilador con datos de simulador.
5. El valor de las PINNs está en resolver lo que los métodos clásicos no resuelven directamente.

---

**Referencias del código.** Moseley, B. (2022). *So, what is a physics-informed neural network?* · Lu, L. et al. (2021). DeepXDE. · Sitzmann, V. et al. (2020). SIREN. · Zill, D. G. (2009). *Ecuaciones diferenciales con aplicaciones de modelado*.

**Contacto:** Lenin Herrera Padilla · lenin.herrera.p@uni.pe